# LoRA Operations Notebook

Interactive notebook for managing LoRA adapter lifecycle:
register, promote, demote, download, and sync adapters.

## Sections
1. Setup & Configuration
2. Inspect Training Runs
3. Register Adapter
4. Promote / Demote
5. Sync to Inference Host

## 1. Setup & Configuration

In [ ]:
import os
import sys
from pathlib import Path

import dotenv

PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", Path("../..").resolve())).resolve()
os.environ["PROJECT_ROOT"] = str(PROJECT_ROOT)

for p in [str(PROJECT_ROOT), str(PROJECT_ROOT / "src")]:
    if p not in sys.path:
        sys.path.insert(0, p)

# Load MLflow connection settings
dotenv.load_dotenv(PROJECT_ROOT / "experiments" / ".env")

import mlflow

tracking_uri = os.getenv("MLFLOW_BACKEND_URI")
if tracking_uri:
    mlflow.set_tracking_uri(tracking_uri)

print(f"PROJECT_ROOT     : {PROJECT_ROOT}")
print(f"MLflow tracking  : {mlflow.get_tracking_uri()}")

## 2. Inspect Training Runs

List recent MLflow runs and their metrics.

In [ ]:
EXPERIMENT_NAME = "train_adapter"

runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    max_results=10,
    order_by=["start_time DESC"],
)

if runs.empty:
    print(f"No runs found in experiment '{EXPERIMENT_NAME}'.")
else:
    cols = [
        c
        for c in ["run_id", "status", "start_time", "metrics.val_loss", "metrics.train_loss_epoch"]
        if c in runs.columns
    ]
    display(runs[cols])

## 3. Register Adapter

Register a trained adapter from an MLflow run into the Model Registry.

In [ ]:
from shared.model_registry import AdapterRegistry

registry = AdapterRegistry()

# Replace <RUN_ID> with the run_id from the table above
RUN_ID = ""  # ← paste run_id here
MODEL_NAME = "lora-summarize"

if RUN_ID:
    registry.register(model_name=MODEL_NAME, run_id=RUN_ID)
    print(f"Registered {MODEL_NAME} from run {RUN_ID}")
else:
    print("Set RUN_ID above to register an adapter.")

## 4. Promote / Demote

Assign or remove aliases (champion, challenger) on registered versions.

In [ ]:
# Assign an alias to a registered model version
# registry.promote(model_name="lora-summarize", version=3, alias="champion")

# Remove an alias
# registry.demote(model_name="lora-summarize", alias="champion")

# List registered versions
versions = registry.list_versions(model_name=MODEL_NAME)
if versions:
    for v in versions:
        print(f"  v{v.version}  aliases={v.aliases}  status={v.status}")
else:
    print(f"No registered versions for '{MODEL_NAME}'.")

## 5. Sync to Inference Host

Download champion adapters and hot-load them into a running vLLM instance.

In [ ]:
# Download champion adapter and sync to inference host
# Uncomment and configure the URL for your vLLM deployment

# registry.sync(
#     adapters_dir=PROJECT_ROOT / "assets" / "adapters",
#     vllm_url="http://localhost:8000",
# )

print("Sync step is manual — uncomment and configure the vLLM URL above.")